# Geopolitics Dataset EDA

This notebook explores the geopolitics slice produced by `scripts/download_resolved_probability_dataset.py` and saved into `db/resolved_probability_dataset.sqlite`.

Focus:
1. validate the SQLite dataset contents,
2. inspect market coverage and row counts,
3. look at sparsity and trade activity in the `5m` panel,
4. visualize probability paths for the largest resolved geopolitics markets.


## Step 0: Environment Setup

This notebook is meant to run from either the repository root or the `examples/` directory.


In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "clients").exists() else cwd.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


Repo root: /Users/sneddy/research/polymarket_research


In [2]:
from __future__ import annotations

import json
import sqlite3

import matplotlib
try:
    from IPython import get_ipython
    if get_ipython() is None:
        matplotlib.use("Agg")
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


## Step 1: Load The SQLite Dataset


In [3]:
DB_PATH = REPO_ROOT / "db" / "resolved_probability_dataset.sqlite"
CATEGORY = "geopolitics"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Dataset DB not found: {DB_PATH}")

print(f"Using DB: {DB_PATH}")
print(f"Category: {CATEGORY}")


Using DB: /Users/sneddy/research/polymarket_research/db/resolved_probability_dataset.sqlite
Category: geopolitics


In [4]:
with sqlite3.connect(DB_PATH) as conn:
    markets_df = pd.read_sql_query(
        "SELECT * FROM markets WHERE primary_domain = ? ORDER BY volume_num DESC, created_at DESC",
        conn,
        params=(CATEGORY,),
    )
    added_df = pd.read_sql_query(
        "SELECT * FROM added_markets WHERE primary_domain = ? ORDER BY added_at_utc DESC",
        conn,
        params=(CATEGORY,),
    )
    probabilities_df = pd.read_sql_query(
        """
        SELECT p.*
        FROM probabilities AS p
        INNER JOIN markets AS m
            ON m.market_id = p.market_id
        WHERE m.primary_domain = ?
        ORDER BY p.timestamp_utc ASC
        """,
        conn,
        params=(CATEGORY,),
    )

for col in ["created_at", "end_date", "synced_at_utc"]:
    if col in markets_df.columns:
        markets_df[col] = pd.to_datetime(markets_df[col], utc=True, errors="coerce")

for col in ["added_at_utc", "probability_start_utc", "probability_end_utc"]:
    if col in added_df.columns:
        added_df[col] = pd.to_datetime(added_df[col], utc=True, errors="coerce")

probabilities_df["timestamp_utc"] = pd.to_datetime(probabilities_df["timestamp_utc"], utc=True, errors="coerce")

for col in ["tag_labels", "matched_tags", "matched_domains"]:
    if col in markets_df.columns:
        markets_df[col] = markets_df[col].map(lambda v: json.loads(v) if isinstance(v, str) and v.startswith("[") else v)

print(f"markets rows: {len(markets_df):,}")
print(f"added_markets rows: {len(added_df):,}")
print(f"probabilities rows: {len(probabilities_df):,}")


markets rows: 1,991
added_markets rows: 1,991
probabilities rows: 24,770,043


In [5]:
probabilities_df.yes_probability

In [14]:
probabilities_df

## Step 2: High-Level Overview


In [5]:
overview_df = pd.DataFrame(
    [
        {
            "n_markets": int(markets_df["market_id"].nunique()),
            "n_added_markets": int(added_df["market_id"].nunique()),
            "n_probability_rows": int(len(probabilities_df)),
            "min_timestamp_utc": probabilities_df["timestamp_utc"].min(),
            "max_timestamp_utc": probabilities_df["timestamp_utc"].max(),
            "mean_yes_probability": float(probabilities_df["yes_probability"].mean()),
            "mean_trade_count_per_bucket": float(probabilities_df["trade_count"].mean()),
            "share_observed_trade_buckets": float(probabilities_df["observed_trade"].mean()),
        }
    ]
)
display(overview_df)


In [6]:
market_view_cols = [
    "market_id",
    "market_slug",
    "question",
    "volume_num",
    "final_outcome",
    "final_yes_probability",
    "matched_domains",
    "matched_tags",
]
display(markets_df[market_view_cols].head(15))


## Step 3: Coverage And Sparsity By Market


In [7]:
per_market_df = (
    probabilities_df.groupby("market_id", as_index=False)
    .agg(
        n_rows=("timestamp_utc", "size"),
        min_timestamp_utc=("timestamp_utc", "min"),
        max_timestamp_utc=("timestamp_utc", "max"),
        mean_yes_probability=("yes_probability", "mean"),
        observed_trade_share=("observed_trade", "mean"),
        total_trades=("trade_count", "sum"),
        total_size=("total_size", "sum"),
    )
    .merge(
        markets_df[["market_id", "market_slug", "question", "volume_num", "final_outcome"]],
        on="market_id",
        how="left",
    )
    .sort_values(["n_rows", "volume_num"], ascending=[False, False])
    .reset_index(drop=True)
)

per_market_df["span_days"] = (
    per_market_df["max_timestamp_utc"] - per_market_df["min_timestamp_utc"]
).dt.total_seconds() / 86400.0

display(per_market_df.head(16))


In [8]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_df = per_market_df.head(10).sort_values("volume_num", ascending=True)
axes[0].barh(plot_df["market_slug"], plot_df["volume_num"], color="#2f6db3")
axes[0].set_title("Top Geopolitics Markets By Volume")
axes[0].set_xlabel("volume_num")

plot_df = per_market_df.head(10).sort_values("n_rows", ascending=True)
axes[1].barh(plot_df["market_slug"], plot_df["n_rows"], color="#b35c2f")
axes[1].set_title("Top Geopolitics Markets By 5m Rows")
axes[1].set_xlabel("n_rows")

plt.tight_layout()
plt.show()


In [9]:
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    per_market_df["n_rows"],
    per_market_df["observed_trade_share"],
    s=np.clip(per_market_df["volume_num"] / 5_000, 20, 500),
    alpha=0.7,
    color="#2a9d8f",
)
for _, row in per_market_df.head(8).iterrows():
    ax.annotate(row["market_slug"], (row["n_rows"], row["observed_trade_share"]), fontsize=8, alpha=0.8)
ax.set_title("Panel Length vs Observed-Trade Share")
ax.set_xlabel("5m rows")
ax.set_ylabel("share of buckets with a fresh trade")
plt.tight_layout()
plt.show()


## Step 4: Probability Path Visualization


In [10]:
top_market_ids = per_market_df.head(6)["market_id"].tolist()
plot_probs_df = probabilities_df[probabilities_df["market_id"].isin(top_market_ids)].copy()
plot_probs_df = plot_probs_df.merge(
    markets_df[["market_id", "market_slug", "final_outcome"]],
    on="market_id",
    how="left",
)

fig, axes = plt.subplots(len(top_market_ids), 1, figsize=(14, 3 * len(top_market_ids)), sharex=False)
if len(top_market_ids) == 1:
    axes = [axes]

for ax, market_id in zip(axes, top_market_ids, strict=False):
    market_slice = plot_probs_df[plot_probs_df["market_id"] == market_id].sort_values("timestamp_utc")
    label = market_slice["market_slug"].iloc[0]
    final_outcome = market_slice["final_outcome"].iloc[0]
    ax.plot(market_slice["timestamp_utc"], market_slice["yes_probability"], lw=1.4, color="#264653")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"{label} | final_outcome={final_outcome}")
    ax.set_ylabel("P(Yes)")

axes[-1].set_xlabel("timestamp_utc")
plt.tight_layout()
plt.show()


## Step 5: One-Market Deep Dive

Pick a single market and inspect its `5m` path together with bucketed trade activity.


In [11]:
selected_market_id = per_market_df.iloc[0]["market_id"]
selected_market_slug = per_market_df.iloc[0]["market_slug"]
selected_market_question = per_market_df.iloc[0]["question"]
selected_market_df = probabilities_df[probabilities_df["market_id"] == selected_market_id].sort_values("timestamp_utc")

print(f"Selected market_id: {selected_market_id}")
print(f"Slug: {selected_market_slug}")
print(f"Question: {selected_market_question}")
print(f"Rows: {len(selected_market_df):,}")

display(selected_market_df.head(12))


Selected market_id: 517817
Slug: will-trump-acquire-greenland-in-2025
Question: Will Trump acquire Greenland in 2025?
Rows: 103,323


In [12]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(selected_market_df["timestamp_utc"], selected_market_df["yes_probability"], color="#1d3557", lw=1.5)
axes[0].set_title(f"Probability Path: {selected_market_slug}")
axes[0].set_ylabel("P(Yes)")
axes[0].set_ylim(-0.02, 1.02)

axes[1].bar(selected_market_df["timestamp_utc"], selected_market_df["trade_count"], width=1/288, color="#e76f51")
axes[1].set_title("Trade Count Per 5m Bucket")
axes[1].set_ylabel("trade_count")
axes[1].set_xlabel("timestamp_utc")

plt.tight_layout()
plt.show()


## Step 6: Final State Sanity Check

The final bucket should generally line up with the resolved outcome for these resolved markets.


In [13]:
last_probability_df = (
    probabilities_df.sort_values(["market_id", "timestamp_utc"])
    .groupby("market_id", as_index=False)
    .tail(1)
    [["market_id", "timestamp_utc", "yes_probability"]]
    .rename(columns={"timestamp_utc": "last_timestamp_utc", "yes_probability": "last_yes_probability"})
)

sanity_df = (
    markets_df[["market_id", "market_slug", "question", "final_outcome", "final_yes_probability"]]
    .merge(last_probability_df, on="market_id", how="left")
    .sort_values("last_yes_probability", ascending=False)
    .reset_index(drop=True)
)

display(sanity_df)
